# Code generation & debugging assistant

Companion notebook for **lesson 5.2**. An LLM writes `process_data`, tests run it in an isolated process, and the full test report goes back to the model until every test passes, or progress stalls.

**Run it in Colab, setup cell first.** It uses Colab's built-in models (no API key); uncomment a `MODEL = ...` line there to choose one.

In [ ]:
# Setup: Colab's built-in models, no API key needed. Run this cell first.
from google.colab import ai
print("Available models:", ", ".join(ai.list_models()))

MODEL = None                              # Colab's default; uncomment a line below to choose
# MODEL = "google/gemini-2.5-flash"
# MODEL = "google/gemini-2.5-pro"
# MODEL = "google/gemini-3.8-flash"

def get_completion(messages):
    """Send chat messages to the model. Colab's generate_text takes one prompt, so roles are inlined."""
    prompt = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in messages) + "\n\nASSISTANT:\n"
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

## 1 · Spec and tests

The spec states every requirement the tests check, so the model doesn't have to reverse-engineer them from failures. In `TESTS`, an exception class as the expected value means "should raise".

In [1]:
TASK = """Write a Python function `process_data(numbers, mode="average")`:
- mode "sum" returns the sum, "average" the mean, "median" the median of the numeric values
- ignore anything that isn't an int or float (booleans don't count as numbers)
- return None when there are no numeric values
- raise ValueError for any other mode
"""

TESTS = [
    (([1, 2, 3, 4, 5], "sum"),             15),
    (([1, 2, 3, 4, 5], "average"),         3.0),
    (([1.5, 2.5], "sum"),                  4.0),
    (([1, 3, 4], "median"),                3),
    (([1, 2, 3, 5], "median"),             2.5),
    (([1, 2, "a", 3], "sum"),              6),
    (([1, 2, None, 3, "b", 4], "average"), 2.5),
    (([True, 2, 3], "sum"),                5),
    (([], "sum"),                          None),
    (([], "median"),                       None),
    (([1, 2, 3], "invalid_mode"),          ValueError),
]

## 2 · The test runner

The candidate code and the tests go to a separate Python process over stdin, with a timeout: a crash, a syntax error or an infinite loop becomes a failed test instead of breaking the notebook. The report lists **every** test, passing ones too, so a fix can't quietly break something that worked.

In [2]:
import json, re, subprocess, sys

CHILD = r"""
import json, sys
job = json.load(sys.stdin)
ns = {}
exec(job["code"], ns)
out = []
for args, expected, raises in job["tests"]:
    try:
        actual = ns["process_data"](*args)
        out.append({"actual": repr(actual), "passed": raises is None and actual == expected})
    except Exception as e:
        out.append({"actual": f"{type(e).__name__}: {e}", "passed": type(e).__name__ == raises})
print(json.dumps(out))
"""

def run_tests(code: str, tests=TESTS) -> list[dict]:
    raises = lambda exp: exp.__name__ if isinstance(exp, type) else None
    wire = [(list(args), None if raises(exp) else exp, raises(exp)) for args, exp in tests]
    try:
        proc = subprocess.run([sys.executable, "-c", CHILD], input=json.dumps({"code": code, "tests": wire}),
                              capture_output=True, text=True, timeout=5)
        if proc.returncode == 0:
            outcomes = json.loads(proc.stdout)
        else:                                             # didn't load: syntax error, crash
            outcomes = [{"actual": proc.stderr.strip().splitlines()[-1], "passed": False}] * len(tests)
    except subprocess.TimeoutExpired:
        outcomes = [{"actual": "timed out", "passed": False}] * len(tests)
    return [{"call": f"process_data{args!r}", "expected": raises(exp) or repr(exp), **o}
            for (args, exp), o in zip(tests, outcomes)]

def report(results: list[dict]) -> str:
    rows = [f"{'PASS' if r['passed'] else 'FAIL'}  {r['call']:<50} expected {r['expected']:<10} got {r['actual']}"
            for r in results]
    return f"{sum(r['passed'] for r in results)}/{len(results)} passed\n" + "\n".join(rows)

## 3 · Try the runner on hand-written candidates

No model yet: a first draft that only knows `sum` and `average` (and uses a plain `isinstance` check), then a complete version. This is the kind of report the model gets.

In [3]:
DRAFT = """
def process_data(numbers, mode="average"):
    values = [n for n in numbers if isinstance(n, (int, float))]
    if mode == "sum":
        return sum(values)
    return sum(values) / len(values)
"""

COMPLETE = """
from statistics import median

def process_data(numbers, mode="average"):
    values = [n for n in numbers if isinstance(n, (int, float)) and not isinstance(n, bool)]
    if mode not in ("sum", "average", "median"):
        raise ValueError(f"unknown mode: {mode!r}")
    if not values:
        return None
    return {"sum": sum, "average": lambda v: sum(v) / len(v), "median": median}[mode](values)
"""

print("── draft\n" + report(run_tests(DRAFT)) + "\n")
print("── complete\n" + report(run_tests(COMPLETE)))

── draft
5/11 passed
PASS  process_data([1, 2, 3, 4, 5], 'sum')               expected 15         got 15
PASS  process_data([1, 2, 3, 4, 5], 'average')           expected 3.0        got 3.0
PASS  process_data([1.5, 2.5], 'sum')                    expected 4.0        got 4.0
FAIL  process_data([1, 3, 4], 'median')                  expected 3          got 2.6666666666666665
FAIL  process_data([1, 2, 3, 5], 'median')               expected 2.5        got 2.75
PASS  process_data([1, 2, 'a', 3], 'sum')                expected 6          got 6
PASS  process_data([1, 2, None, 3, 'b', 4], 'average')   expected 2.5        got 2.5
FAIL  process_data([True, 2, 3], 'sum')                  expected 5          got 6
FAIL  process_data([], 'sum')                            expected None       got 0
FAIL  process_data([], 'median')                         expected None       got ZeroDivisionError: division by zero
FAIL  process_data([1, 2, 3], 'invalid_mode')            expected ValueError got 2.0

──

## 4 · Pulling code out of the reply

Models wrap code in Markdown fences, but not always the same way. Take the first fenced block (`python`, `py` or bare), or the whole reply if there's no fence.

In [4]:
def extract_code(reply: str) -> str:
    match = re.search(r"`{3}[ \t]*(?:python|py)?[ \t]*\n(.*?)`{3}", reply, re.S)
    return (match.group(1) if match else reply).strip()

fence = "`" * 3
for reply in [f"{fence}python\ndef f(): pass\n{fence}",
              f"{fence}python \ndef f(): pass\n{fence}",
              f"Here you go:\n{fence}py\ndef f(): pass\n{fence}\nHope it helps!",
              "def f(): pass"]:
    print(repr(extract_code(reply)))

'def f(): pass'
'def f(): pass'
'def f(): pass'
'def f(): pass'


## 5 · The loop

Write → test → send the full report back → repeat, until everything passes, the iteration limit is hit, or the pass count hasn't improved for `patience` rounds (then more of the same won't help). Each attempt is recorded for monitoring.

In [5]:
from dataclasses import dataclass

@dataclass
class Attempt:
    iteration: int
    passed: int
    total: int
    code: str

def ask(prompt: str) -> str:
    return extract_code(get_completion([{"role": "user", "content": prompt}]))

def refine(max_iterations: int = 5, patience: int = 2) -> list[Attempt]:
    attempts: list[Attempt] = []
    code = ask(f"{TASK}\nReturn only the function in a ```python code block.")
    for i in range(max_iterations):
        results = run_tests(code)
        attempts.append(Attempt(i, sum(r["passed"] for r in results), len(results), code))
        print(f"Iteration {i}: {attempts[-1].passed}/{attempts[-1].total} passed")
        if attempts[-1].passed == attempts[-1].total:
            print("✅ All tests pass.")
            break
        best_before = max((a.passed for a in attempts[:-patience]), default=-1)
        if len(attempts) > patience and max(a.passed for a in attempts[-patience:]) <= best_before:
            print(f"⏹ No progress in {patience} iterations: change the prompt, model or tests.")
            break
        code = ask(f"{TASK}\nCurrent code:\n```python\n{code}\n```\n"
                   f"Test report (keep passing tests passing):\n{report(results)}\n\n"
                   f"Return only the corrected function in a ```python code block.")
    return attempts

## 6 · Run it

Uses Colab's built-in model; results vary between runs.

In [ ]:
attempts = refine()
print("\nFinal code:\n" + attempts[-1].code)